In [ ]:
!pip install --quiet kagglehub[pandas-datasets]

import kagglehub
from pathlib import Path

# Descarga completa del dataset
DATASET_DIR = kagglehub.dataset_download("ambityga/imagenet100")

print("Archivos descargados en:", DATASET_DIR)

LABELS_PATH = Path(DATASET_DIR) / "Labels.json"
VAL_DIR = Path(DATASET_DIR) / "val.X"
print(VAL_DIR)
TRAIN1_DIR = Path(DATASET_DIR)/"train.X1"
TRAIN2_DIR = Path(DATASET_DIR)/"train.X2"
TRAIN3_DIR = Path(DATASET_DIR)/"train.X3"
TRAIN4_DIR = Path(DATASET_DIR)/"train.X4"

Using Colab cache for faster access to the 'imagenet100' dataset.
Archivos descargados en: /kaggle/input/imagenet100
/kaggle/input/imagenet100/val.X


In [ ]:
from google.colab import files

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import ResNet34_Weights
import json
import torch.nn.functional as F
class ImageNet100ValDataset(Dataset):
    def __init__(self, root_dir, transform=None, labels_json=LABELS_PATH):
        self.root_dir = root_dir
        self.transform = transform

        # Cargar mapeo global desde Labels.json
        with open(labels_json) as f:
            labels = json.load(f)

        # Usar el orden y mapeo original de índices
        self.class_to_idx = {wnid: i for i, wnid in enumerate(labels.keys())}

        # Cargar las muestras
        self.samples = []
        for wnid in self.class_to_idx:
            class_dir = os.path.join(root_dir, wnid)
            if not os.path.isdir(class_dir):
                continue
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(class_dir, f), self.class_to_idx[wnid]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

MEAN_DATASET = [0.485, 0.456, 0.406]
STD_DATASET  = [0.229, 0.224, 0.225]
transform = transforms.Compose([
    transforms.Resize(256),                  # Redimensiona el lado más corto a 256 px
    transforms.CenterCrop(224),              # Recorta el centro a 224×224 (tamaño típico de ImageNet)
    transforms.ToTensor(),                   # Convierte a tensor (0–1)
    transforms.Normalize(                    # Normaliza con medias y desv. estándar de ImageNet
        mean= MEAN_DATASET,
        std =STD_DATASET
    )
])


with open(LABELS_PATH) as f:
    labels = json.load(f)

selected_classes = list(labels.keys())

weights = ResNet34_Weights.DEFAULT

imagenet_classes = weights.meta["categories"]

# Mapear WNID a nombre de clase entendible por el modelo
wnid_to_name = {wnid: labels[wnid].split(',')[0] for wnid in selected_classes}

# Obtener índices dentro de las 1000 clases del modelo
selected_indices_in_model = [imagenet_classes.index(wnid_to_name[wnid]) for wnid in selected_classes]
print(selected_indices_in_model)
# ------------------ Estadísticas de ImageNet ------------------
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]


# ------------------ utilidades ------------------
def denormalize_tensor(tensor_norm, mean=MEAN, std=STD):
    """tensor_norm: normalizado -> pixel-space [0,1]"""
    m = torch.tensor(mean, device=tensor_norm.device).view(1, -1, 1, 1)
    s = torch.tensor(std,  device=tensor_norm.device).view(1, -1, 1, 1)
    return (tensor_norm * s + m).clamp(0.0, 1.0)

def normalize_tensor(px, mean=MEAN, std=STD):
    m = torch.tensor(mean, device=px.device).view(1, -1, 1, 1)
    s = torch.tensor(std,  device=px.device).view(1, -1, 1, 1)
    return (px - m) / s

def evaluar_imagen(model, img, selected_indices_in_model):
    """
    Evalúa una sola imagen en el modelo.

    Parámetros:
        model: modelo preentrenado (por ejemplo, resnet50)
        img: tensor de imagen (C, H, W) ya transformado
        selected_indices_in_model: lista de índices de las clases que se quieren evaluar en el modelo

    Devuelve:
        pred_wnid: WNID predicho
        nombre_legible: nombre de la clase predicha
        prob: probabilidad asociada
    """

    input_tensor = img.unsqueeze(0)

    # Inferencia sin gradientes
    with torch.no_grad():
        output = model(input_tensor)

        # Filtrar los logits solo para las clases seleccionadas
        filtered_logits = output[0][selected_indices_in_model]
        filtered_probs = torch.nn.functional.softmax(filtered_logits, dim=0)

        # Elegir la clase más probable
        pred_idx_in_filtered = filtered_probs.argmax().item()
        pred_wnid = selected_classes[pred_idx_in_filtered]
        prob = filtered_probs[pred_idx_in_filtered].item()

        nombre_legible = labels[pred_wnid]

    return pred_wnid, nombre_legible, prob

def load_resnet34(device=None):
    """
    Carga un modelo ResNet34 preentrenado con pesos de ImageNet,
    listo para evaluación, junto con su lista de clases.
    """
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    weights = ResNet34_Weights.DEFAULT
    model = resnet34(weights=weights).to(device).eval()
    imagenet_classes = weights.meta["categories"]
    preprocess = weights.transforms()
    return model, imagenet_classes, preprocess


# =====================================================
# MAPEO ENTRE WNID Y CLASES DEL MODELO
# =====================================================

def load_wnid_mapping(labels_json_path=LABELS_PATH):
    """Carga el mapeo WNID → nombre legible desde un JSON."""
    with open(labels_json_path, "r") as f:
        wnid2name = json.load(f)
    return wnid2name


def build_class_mapping(dataset_root, imagenet_classes, wnid2name):
    """
    Construye un mapeo robusto entre las carpetas locales de ImageNet100
    y los índices de clase del modelo ResNet34 (1000 clases).

    Retorna:
        wnid_to_model_idx, model_idx_to_wnid, selected_indices_in_model
    """
    dataset_root = Path(dataset_root)
    selected_classes = sorted([p.name for p in dataset_root.iterdir() if p.is_dir()])

    wnid_to_model_idx = {}
    for wnid in selected_classes:
        if wnid not in wnid2name:
            continue
        wnid_name = wnid2name[wnid].split(",")[0].lower().strip()
        match = [i for i, c in enumerate(imagenet_classes) if wnid_name in c.lower()]
        if match:
            wnid_to_model_idx[wnid] = match[0]

    selected_indices_in_model = list(wnid_to_model_idx.values())
    model_idx_to_wnid = {v: k for k, v in wnid_to_model_idx.items()}

    print(f"Total carpetas detectadas en {dataset_root}: {len(selected_classes)}")
    print(f"Total clases mapeadas correctamente: {len(selected_indices_in_model)}")

    if not selected_indices_in_model:
        raise ValueError("No se encontró ninguna clase del dataset en las 1000 del modelo. Verifica Labels.json.")

    return wnid_to_model_idx, model_idx_to_wnid, selected_indices_in_model


# =====================================================
# EVALUACIÓN TOP-5 LIMITADA A 100 CLASES
# =====================================================

def top5_for_image_path(model, img_path, preprocess, imagenet_classes,
                        selected_indices_in_model, model_idx_to_wnid, wnid2name,
                        device=None):
    """
    Evalúa una imagen (ruta) y retorna las top-5 predicciones restringidas
    a las clases seleccionadas del dataset (p.ej. las 100 de ImageNet100).

    Devuelve una lista de dicts con:
    - model_idx
    - class_name
    - prob
    - wnid
    - wnid_name
    """
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    img = Image.open(img_path).convert("RGB")
    x = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)[0]
        logits_filtered = logits[selected_indices_in_model]
        probs = F.softmax(logits_filtered, dim=0)

    k = min(5, len(selected_indices_in_model))
    topk = torch.topk(probs, k=k)

    results = []
    for idx_f, p in zip(topk.indices.cpu().tolist(), topk.values.cpu().tolist()):
        model_idx = selected_indices_in_model[idx_f]
        wnid = model_idx_to_wnid.get(model_idx, "unknown")
        wnid_name = wnid2name.get(wnid, "desconocido")
        class_name = imagenet_classes[model_idx]
        results.append({
            "model_idx": model_idx,
            "class_name": class_name,
            "prob": p,
            "wnid": wnid,
            "wnid_name": wnid_name
        })
    return results


def wnid_to_model_index(wnid, weights, labels_json=LABELS_PATH):
    """
    Devuelve el índice (0..999) en weights.meta['categories'] correspondiente al wnid.
    Intenta coincidencia exacta con el nombre "primera parte" del Labels.json,
    y si no encuentra intenta coincidencia parcial.
    Lanza ValueError si no puede mapear.
    """
    with open(labels_json, "r") as f:
        wnid2name = json.load(f)

    if wnid not in wnid2name:
        raise ValueError(f"WNID {wnid} no está en {labels_json}")

    target_name = wnid2name[wnid].split(",")[0].strip().lower()  # p.ej. "wombat"
    imagenet_classes = weights.meta["categories"]

    # 1) buscar coincidencia exacta (comparando nombres lower)
    for i, cname in enumerate(imagenet_classes):
        if cname.lower().strip() == target_name:
            return i

    # 2) intentar coincidencia parcial por tokens
    target_token = target_name.split()[0]
    for i, cname in enumerate(imagenet_classes):
        if target_token in cname.lower():
            return i

    raise ValueError(f"No pude mapear {wnid} -> índice en weights.meta['categories'] (buscado '{target_name}').")

def global_evaluate(model, metodo, param, VAL_DIR="val.X"):

    model.eval()
    val_dataset = ImageNet100ValDataset(VAL_DIR, transform=transform, labels_json=LABELS_PATH)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    correct_top1 = 0
    correct_top5 = 0
    total = 0

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for imgs, labels_idx in val_loader:

        imgs = imgs.to(device)
        labels_idx = labels_idx.to(device)

        if metodo is None:
            att_imgs = imgs
        else:
            att_imgs = torch.stack([
                metodo(model, imgs[i].to(device), labels_idx[i].to(device), param)
                for i in range(len(imgs))
            ]).to(device)

        with torch.no_grad():
            outputs = model(att_imgs)

            filtered_logits = outputs[:, selected_indices_in_model]
            filtered_probs = F.softmax(filtered_logits, dim=1)

            preds_in_filtered = filtered_probs.argmax(dim=1)
            top5_preds = torch.topk(filtered_probs, 5, dim=1).indices

            pred_wnids = [selected_classes[i] for i in preds_in_filtered]
            true_wnids = [list(val_dataset.class_to_idx.keys())[i] for i in labels_idx]


            for i in range(labels_idx.size(0)):
                if labels_idx[i].item() in top5_preds[i]:
                    correct_top5 += 1

            correct_top1 += sum(p == t for p, t in zip(pred_wnids, true_wnids))
            total += len(imgs)

    return correct_top1/total, correct_top5/total


def global_transfer(model_atk, model_trans, metodo, param, VAL_DIR="val.X"):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_atk.to(device).eval()
    model_trans.to(device).eval()
    val_dataset = ImageNet100ValDataset(VAL_DIR, transform=transform, labels_json=LABELS_PATH)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    correct_top1 = 0
    correct_top5 = 0
    total = 0


    for imgs, labels_idx in val_loader:

        imgs = imgs.to(device)
        labels_idx = labels_idx.to(device)

        if metodo is None:
            att_imgs = imgs
        else:
            att_imgs = torch.stack([
                metodo(model_atk, imgs[i].to(device), labels_idx[i].to(device), param)

                for i in range(len(imgs))
            ]).to(device)

        with torch.no_grad():
            outputs = model_trans(att_imgs)


            filtered_logits = outputs[:, selected_indices_in_model]
            filtered_probs = F.softmax(filtered_logits, dim=1)

            preds_in_filtered = filtered_probs.argmax(dim=1)
            top5_preds = torch.topk(filtered_probs, 5, dim=1).indices

            pred_wnids = [selected_classes[i] for i in preds_in_filtered]
            true_wnids = [list(val_dataset.class_to_idx.keys())[i] for i in labels_idx]


            for i in range(labels_idx.size(0)):
                if labels_idx[i].item() in top5_preds[i]:
                    correct_top5 += 1

            correct_top1 += sum(p == t for p, t in zip(pred_wnids, true_wnids))
            total += len(imgs)

    return correct_top1/total, correct_top5/total



[117, 70, 88, 133, 5, 97, 42, 60, 14, 3, 130, 57, 26, 0, 89, 127, 36, 67, 110, 65, 123, 55, 22, 21, 1, 71, 99, 16, 19, 108, 18, 35, 124, 90, 74, 129, 125, 2, 64, 92, 138, 48, 54, 39, 56, 96, 84, 73, 77, 52, 20, 118, 111, 59, 106, 75, 143, 80, 140, 11, 113, 4, 28, 50, 38, 104, 24, 107, 100, 81, 94, 41, 68, 8, 66, 146, 29, 32, 137, 33, 141, 517, 78, 150, 76, 61, 112, 83, 144, 91, 135, 116, 72, 34, 6, 119, 46, 115, 93, 7]


In [ ]:
def ensure_batch(imgs, labels):
    """
    imgs   : (C,H,W) o (B,C,H,W)
    labels : escalar o tensor(B)
    Devuelve imgs_batched, labels_batched, single_input_flag
    """
    single = False

    if imgs.dim() == 3:        # single (C,H,W)
        imgs = imgs.unsqueeze(0)
        labels = torch.tensor([labels], device=imgs.device)
        single = True

    return imgs, labels, single

def generar_imagen_gaussian(model, imgs, labels, args):
    sigma = args[0]

    imgs, labels, single = ensure_batch(imgs, labels)

    noise = torch.randn_like(imgs) * sigma
    adv = imgs + noise
    adv = adv.clamp(-3, 3)

    return adv[0] if single else adv
# @title
def generar_imagen_rfgsm(model, imgs, labels, args):
    eps  = args[0]
    alpha = args[1]

    imgs, labels, single = ensure_batch(imgs, labels)

    noise = torch.empty_like(imgs).uniform_(-alpha, alpha)
    grad = image_gradient(model, imgs + noise, labels)

    adv = imgs + noise - eps * grad.sign()
    adv = adv.clamp(-3, 3)

    return adv[0] if single else adv
# @title

def generar_imagen_PGD(model, imgs, labels, args):
    eps     = args[0]
    alpha   = args[1]
    iters   = args[2]

    imgs, labels, single = ensure_batch(imgs, labels)

    # Inicialización aleatoria dentro de [-eps,eps]
    noise = torch.empty_like(imgs).uniform_(-eps, eps)
    adv = imgs + noise

    for _ in range(iters):
        grad = image_gradient(model, adv, labels)
        adv = adv - alpha * grad.sign()

        # Proyecto al L∞ (imagenes normalizadas)
        perturb = torch.clamp(adv - imgs, -eps, eps)
        adv = imgs + perturb

    return adv[0] if single else adv

def generar_imagen_RPGD(model, imgs, labels, args):
    eps     = args[0]
    alpha   = args[1]
    iters   = args[2]
    sigma   = args[3]

    imgs, labels, single = ensure_batch(imgs, labels)

    noise = torch.empty_like(imgs).uniform_(-eps, eps)
    adv = imgs + noise

    for _ in range(iters):
        grad = image_gradient(model, adv, labels)
        adv = adv - alpha * grad.sign()

        perturb = torch.clamp(adv - imgs, -eps, eps)
        adv = imgs + perturb

        # ruido aleatorio al final de cada paso
        noise = torch.empty_like(imgs).uniform_(-sigma, sigma)
        adv = adv + noise

    adv = adv.clamp(-3, 3)
    return adv[0] if single else adv

def generar_imagen_fgsm(model, imgs, labels, args):
    imgs, labels, single = ensure_batch(imgs, labels)
    grad = image_gradient(model, imgs, labels)
    eps = args[0]

    adv = imgs - eps * grad.sign()
    adv = adv.clamp(-3, 3)
    return adv[0] if single else adv



def image_gradient(model, img, labels):
    img = img.clone().detach().to(device)
    img.requires_grad_(True)

    out = model(img)
    loss = F.cross_entropy(out, labels)

    loss.backward()

    grad = img.grad.detach()
    return grad

In [ ]:
mapping_100_to_1000 = {i: cls for i, cls in enumerate(selected_indices_in_model)}
def labels_100_to_1000(labels_100):
    return torch.tensor(
        [mapping_100_to_1000[int(l)] for l in labels_100],
        device=labels_100.device
    )

def elegir_ataque():
    """Devuelve el nombre del ataque elegido al azar."""
    ataques_lista = list(attack_params.keys())
    probs = torch.tensor(list(attack_params.values()), dtype=torch.float)
    idx = torch.multinomial(probs, 1).item()
    return ataques_lista[idx]

def aplicar_ataque(model, img, label):
    ataque = elegir_ataque()

    if ataque == "clean":
        return img

    if ataque == "fgsm":
        return generar_imagen_fgsm(model, img, label, args=[0.05])

    if ataque == "rfgsm":
        return generar_imagen_rfgsm(model, img, label, args=[0.05, 0.01])

    if ataque == "pgd":
        return generar_imagen_PGD(model, img, label, args=[0.05, 0.01, 3])

    if ataque == "rpgd":
        return generar_imagen_RPGD(model, img, label, args=[0.05, 0.01, 3, 0.01])

    if ataque == "gauss":
        return generar_imagen_gaussian(model, img, label, args=[0.05])

    print(f"No esta el ataque {ataque} en esta funcion")
    return img



def evaluar(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)
            preds = outputs[:, selected_indices_in_model].argmax(1)
            correct += (preds == labels).sum().item()
            total += len(imgs)

    return correct / total

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset
from torchvision.models import resnet34, ResNet34_Weights
import matplotlib.pyplot as plt
from torch.utils.data import Subset


val_ds = ImageNet100ValDataset(VAL_DIR, transform=transform)


model = resnet34(weights=None)
state = torch.load("resnet34_vsbalance.pth", map_location="cpu")
model.load_state_dict(state)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory

In [ ]:
print("Acc Val.X Puro")
print(global_evaluate(model, None, [0.05], VAL_DIR))
print("Acc gaussiano")
print(global_evaluate(model, generar_imagen_gaussian, [0.05], VAL_DIR))
print("Acc Fgsm")
print(global_evaluate(model, generar_imagen_fgsm, [0.05], VAL_DIR))
print("Acc Rfgsm")
print(global_evaluate(model, generar_imagen_rfgsm, [0.05, 0.01], VAL_DIR))
print("Acc PGD")
print(global_evaluate(model, generar_imagen_PGD, [0.05, 0.01, 3, 0.01], VAL_DIR))
print("Acc RPgd")
print(global_evaluate(model, generar_imagen_RPGD, [0.05, 0.01, 3, 0.01], VAL_DIR))

Acc Val.X Puro
(0.8118, 0.955)
Acc gaussiano
(0.812, 0.9554)
Acc Fgsm
(0.3868, 0.665)
Acc Rfgsm
(0.3854, 0.6648)
Acc PGD
(0.5928, 0.8708)
Acc RPgd
(0.5984, 0.8728)


In [ ]:
val_ds = ImageNet100ValDataset(VAL_DIR, transform=transform)


model = resnet34(weights=None)
state = torch.load("resnet34_vsfgsm (2).pth", map_location="cpu")
model.load_state_dict(state)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
print("Acc Val.X Puro")
print(global_evaluate(model, None, [0.05], VAL_DIR))
print("Acc gaussiano")
print(global_evaluate(model, generar_imagen_gaussian, [0.05], VAL_DIR))
print("Acc Fgsm")
print(global_evaluate(model, generar_imagen_fgsm, [0.05], VAL_DIR))
print("Acc Rfgsm")
print(global_evaluate(model, generar_imagen_rfgsm, [0.05, 0.01], VAL_DIR))
print("Acc PGD")
print(global_evaluate(model, generar_imagen_PGD, [0.05, 0.01, 3, 0.01], VAL_DIR))
print("Acc RPgd")
print(global_evaluate(model, generar_imagen_RPGD, [0.05, 0.01, 3, 0.01], VAL_DIR))

Acc Val.X Puro
(0.8148, 0.9558)
Acc gaussiano
(0.8148, 0.956)
Acc Fgsm
(0.378, 0.6566)
Acc Rfgsm
(0.3778, 0.6552)
Acc PGD
(0.5486, 0.8542)
Acc RPgd
(0.5546, 0.8598)


In [ ]:
val_ds = ImageNet100ValDataset(VAL_DIR, transform=transform)


model = resnet34(weights=None)
state = torch.load("resnet34_vspgd.pth", map_location="cpu")
model.load_state_dict(state)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
print("Acc Val.X Puro")
print(global_evaluate(model, None, [0.05], VAL_DIR))
print("Acc gaussiano")
print(global_evaluate(model, generar_imagen_gaussian, [0.05], VAL_DIR))
print("Acc Fgsm")
print(global_evaluate(model, generar_imagen_fgsm, [0.05], VAL_DIR))
print("Acc Rfgsm")
print(global_evaluate(model, generar_imagen_rfgsm, [0.05, 0.01], VAL_DIR))
print("Acc PGD")
print(global_evaluate(model, generar_imagen_PGD, [0.05, 0.01, 3, 0.01], VAL_DIR))
print("Acc RPgd")
print(global_evaluate(model, generar_imagen_RPGD, [0.05, 0.01, 3, 0.01], VAL_DIR))

Acc Val.X Puro
(0.8234, 0.9588)
Acc gaussiano
(0.8214, 0.958)
Acc Fgsm
(0.2874, 0.5564)
Acc Rfgsm
(0.2834, 0.556)
Acc PGD
(0.3304, 0.6874)
Acc RPgd
(0.3466, 0.7002)


In [ ]:
model = resnet34(weights=None)
state = torch.load("resnet34_vsbalance.pth", map_location="cpu")
model.load_state_dict(state)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Acc PGD")
print(global_evaluate(model, generar_imagen_PGD, [0.05, 0.01, 5, 0.01], VAL_DIR))
print("Acc RPgd")
print(global_evaluate(model, generar_imagen_RPGD, [0.05, 0.01, 5, 0.01], VAL_DIR))

Acc PGD
(0.441, 0.7724)
Acc RPgd
(0.453, 0.7834)


In [ ]:
model = resnet34(weights=None)
state = torch.load("resnet34_vsfgsm (2).pth", map_location="cpu")
model.load_state_dict(state)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Acc PGD")
print(global_evaluate(model, generar_imagen_PGD, [0.05, 0.01, 5, 0.01], VAL_DIR))
print("Acc RPgd")
print(global_evaluate(model, generar_imagen_RPGD, [0.05, 0.01, 5, 0.01], VAL_DIR))

Acc PGD
(0.3774, 0.7282)
Acc RPgd
(0.3926, 0.7398)


In [ ]:
model = resnet34(weights=None)
state = torch.load("resnet34_vspgd.pth", map_location="cpu")
model.load_state_dict(state)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Acc PGD")
print(global_evaluate(model, generar_imagen_PGD, [0.05, 0.01, 5, 0.01], VAL_DIR))
print("Acc RPgd")
print(global_evaluate(model, generar_imagen_RPGD, [0.05, 0.01, 5, 0.01], VAL_DIR))

Acc PGD
(0.1352, 0.4864)
Acc RPgd
(0.1392, 0.502)


In [ ]:
model = resnet34(weights=None)
state = torch.load("resnet34_vsbalance.pth", map_location="cpu")
model.load_state_dict(state)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Acc PGD")
print(global_evaluate(model, generar_imagen_PGD, [0.05, 0.01, 7, 0.01], VAL_DIR))
print("Acc RPgd")
print(global_evaluate(model, generar_imagen_RPGD, [0.05, 0.01, 7, 0.01], VAL_DIR))

Acc PGD
(0.3256, 0.684)
Acc RPgd
(0.3428, 0.6968)


In [ ]:
model = resnet34(weights=None)
state = torch.load("resnet34_vsfgsm (2).pth", map_location="cpu")
model.load_state_dict(state)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Acc PGD")
print(global_evaluate(model, generar_imagen_PGD, [0.05, 0.01, 7, 0.01], VAL_DIR))
print("Acc RPgd")
print(global_evaluate(model, generar_imagen_RPGD, [0.05, 0.01, 7, 0.01], VAL_DIR))

Acc PGD
(0.2532, 0.625)
Acc RPgd
(0.2692, 0.6444)


In [ ]:
model = resnet34(weights=None)
state = torch.load("resnet34_vspgd.pth", map_location="cpu")
model.load_state_dict(state)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Acc PGD")
print(global_evaluate(model, generar_imagen_PGD, [0.05, 0.01, 7, 0.01], VAL_DIR))
print("Acc RPgd")
print(global_evaluate(model, generar_imagen_RPGD, [0.05, 0.01, 7, 0.01], VAL_DIR))

Acc PGD
(0.0708, 0.3692)
Acc RPgd
(0.07, 0.3984)


In [ ]:
model = resnet34(weights=None)
state = torch.load("resnet34_vsbalance.pth", map_location="cpu")
model.load_state_dict(state)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Acc PGD")
print(global_evaluate(model, generar_imagen_PGD, [0.05, 0.01, 10, 0.01], VAL_DIR))
print("Acc RPgd")
print(global_evaluate(model, generar_imagen_RPGD, [0.05, 0.01, 10, 0.01], VAL_DIR))

Acc PGD
(0.2266, 0.5968)
Acc RPgd
(0.2408, 0.6176)


In [ ]:
model = resnet34(weights=None)
state = torch.load("resnet34_vsfgsm (2).pth", map_location="cpu")
model.load_state_dict(state)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Acc PGD")
print(global_evaluate(model, generar_imagen_PGD, [0.05, 0.01, 10, 0.01], VAL_DIR))
print("Acc RPgd")
print(global_evaluate(model, generar_imagen_RPGD, [0.05, 0.01, 10, 0.01], VAL_DIR))

Acc PGD
(0.1668, 0.531)
Acc RPgd
(0.1844, 0.5512)


In [ ]:
model = resnet34(weights=None)
state = torch.load("resnet34_vspgd.pth", map_location="cpu")
model.load_state_dict(state)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Acc PGD")
print(global_evaluate(model, generar_imagen_PGD, [0.05, 0.01, 10, 0.01], VAL_DIR))
print("Acc RPgd")
print(global_evaluate(model, generar_imagen_RPGD, [0.05, 0.01, 10, 0.01], VAL_DIR))

Acc PGD
(0.039, 0.2682)
Acc RPgd
(0.0422, 0.2796)
